# 02 – Host Economy

- **Research Question:** Is the Airbnb market still a sharing economy driven by local residents renting out single listings, or has it become dominated by real estate agencies and corporate megahosts managing hundreds of apartments?
- **Variables:** `calculated_host_listings_count`, `host_id`, `room_type`, `review_scores_rating`, `review_scores_cleanliness`, `review_scores_communication`
- **Statistical Tools:**
   - Categorical segmentation and Frequency Distribution:
     - Segment binning of listing counts into discrete host categories: Single-host (1), Multi-host (2–4), and Megahost (5+).
     - Cross-tabulation tables calculating normalized relative shares across cities and room types.
     - Market concentration and percentile analysis.
     - Pareto-style concentration analysis measuring the market share controlled by the top 5% largest host accounts.
   - Measures of Central Tendency:
     - Sample Median: Used for review ratings (cleanliness and communication) to prevent skewness from heavy ceiling effects at 5.00.
   - Hypothesis Testing:
     - Mann-Whitney U test: Non-parametric test comparing rating distributions between Single-hosts and Megahosts without assuming normal distribution.
   - Visualizations:
     - Normalized 100% stacked bar charts showing host market structure per city.
     - Grouped bar charts showing room type composition by host segment.
     - Side-by-side boxplots showing medians and IQR dispersion for review scores.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from src.cities import CITIES
from src.load import load_all

# Load listings data with required columns across all 6 cities
cols = [
    'id', 'host_id', 'calculated_host_listings_count', 'room_type', 
    'review_scores_rating', 'review_scores_cleanliness', 
    'review_scores_communication'
]
data = load_all(usecols=cols)

cities_map = {}

for key, city_obj in CITIES.items():
    cities_map[key] = city_obj.name

# Adding city names so they match exactly
data['city_name'] = data['city'].map(cities_map)

num_offers = len(data)
num_cities = data['city'].nunique()

print(f"Loaded {num_offers} offers in {num_cities} cities.")

In [ ]:
# Segment hosts based on calculated_host_listings_count
def assign_segment(cnt):
    if cnt == 1:
        return 'Single-host (1)'
    elif 2 <= cnt <= 4:
        return 'Multi-host (2-4)'
    else:
        return 'Megahost (5+)'

data['host_segment'] = data['calculated_host_listings_count'].apply(assign_segment)
segment_order = ['Single-host (1)', 'Multi-host (2-4)', 'Megahost (5+)']

# Table 1: listing Share (%) by Host Segment per City
city_order = ['Barcelona', 'Los Angeles', 'Madrid', 'New York', 'Tokyo', 'Mexico City']
listings_by_city_seg = pd.crosstab(data['city_name'], data['host_segment'], normalize='index')[segment_order] * 100
listings_by_city_seg = listings_by_city_seg.reindex(city_order)
listings_by_city_seg.round(2)

In [ ]:
# Top 5% hosts market share per city
top5_results = []
for city_key, city_info in CITIES.items():
    city_data = data[data['city'] == city_key]
    host_counts = city_data.groupby('host_id').size().sort_values(ascending=False)
    n_hosts = len(host_counts)
    top5_count = max(1, int(round(n_hosts * 0.05)))
    top5_listings = host_counts.iloc[:top5_count].sum()
    pct_listings = (top5_listings / len(city_data)) * 100
    top5_results.append({
        'City': city_info.name,
        'Total Hosts': n_hosts,
        'Top 5% Host Count': top5_count,
        'Total Listings': len(city_data),
        'Top 5% Listings Controlled': top5_listings,
        'Top 5% Market Share (%)': round(pct_listings, 2)
    })

data_top5 = pd.DataFrame(top5_results).sort_values('Top 5% Market Share (%)', ascending=False)
data_top5

In [ ]:
palette_segment = {
    'Single-host (1)': '#0000FF',
    'Multi-host (2-4)': '#FF5500',
    'Megahost (5+)': '#FF0000'
}

# Plot 1: Host Market Concentration & Share
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

listings_by_city_seg.plot(
    kind='bar', stacked=True, color=[palette_segment[s] for s in segment_order], ax=ax1, width=0.5
)
ax1.set_title('Listing share by host segment per city', fontsize=13, fontweight='bold', pad=13)
ax1.set_ylabel('Percentage of Total Listings (%)', fontsize=13)
ax1.set_xlabel('')
ax1.set_xticklabels(listings_by_city_seg.index, rotation=0, fontsize=13)
ax1.set_ylim(0, 100)
ax1.legend(title='Host Segment', loc='upper right', frameon=True)

for c_idx, city in enumerate(listings_by_city_seg.index):
    cum_h = 0
    for seg in segment_order:
        val = listings_by_city_seg.loc[city, seg]
        if val > 5:
            ax1.text(c_idx, cum_h + val / 2, f"{val:.1f}%", ha='center', va='center', color='white', fontweight='bold', fontsize=10)
        cum_h += val

data_top5_sorted = data_top5.sort_values('Top 5% Market Share (%)', ascending=True)
bars = ax2.barh(data_top5_sorted['City'], data_top5_sorted['Top 5% Market Share (%)'], color='#008000', height=0.5)
ax2.set_title('Market Share Controlled by Top 5% Largest Hosts', fontsize=13, fontweight='bold', pad=12)
ax2.set_xlabel('Share of Listings Controlled by Top 5% Hosts (%)', fontsize=13)
ax2.set_xlim(0, 60)

for bar in bars:
    w = bar.get_width()
    ax2.text(w + 1.0, bar.get_y() + bar.get_height()/2, f"{w:.1f}%", ha='left', va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('../figures/study2_host_concentration.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 2: Room Type Breakdown by Host Segment
fig, ax = plt.subplots(figsize=(10, 5))
room_seg = pd.crosstab(data['host_segment'], data['room_type'], normalize='index')[
    ['Entire home/apt', 'Private room', 'Shared room', 'Hotel room']
] * 100
room_seg = room_seg.reindex(segment_order)

colors_room = ['#1d3557', '#457b9d', '#e63946', '#f4a261']
room_seg.plot(kind='bar', ax=ax, color=colors_room, width=0.7)

ax.set_title('Room Type Distribution by Host Segment (All Cities)', fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel('Percentage within Host Segment (%)', fontsize=13)
ax.set_xlabel('')
ax.set_xticklabels(segment_order, rotation=0, fontsize=10, fontweight='bold')
ax.set_ylim(0, 90)
ax.legend(title='Room Type', frameon=True)

for p in ax.patches:
    h = p.get_height()
    if h > 2:
        ax.annotate(f"{h:.1f}%", (p.get_x() + p.get_width() / 2., h + 1.2),
                    ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/study2_room_type_by_segment.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Ratings Comparison: Cleanliness & Communication
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(
    data=data, x='host_segment', y='review_scores_cleanliness', order=segment_order,
    hue='host_segment', legend=False,
    palette=palette_segment, ax=ax1, showfliers=False, width=0.5
)
ax1.set_title('Cleanliness Rating by Host Segment', fontsize=12, fontweight='bold')
ax1.set_ylabel('Cleanliness Rating', fontsize=11)
ax1.set_xlabel('')
ax1.set_ylim(4.0, 5.08)

clean_medians = data.groupby('host_segment')['review_scores_cleanliness'].median().reindex(segment_order)
for idx, (seg, med) in enumerate(clean_medians.items()):
    ax1.text(idx, med + 0.02, f"Median: {med:.2f}", ha='center', va='bottom', fontweight='bold', color='#111111', bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

u_clean, p_clean = stats.mannwhitneyu(
    data[data['host_segment'] == 'Single-host (1)']['review_scores_cleanliness'].dropna(),
    data[data['host_segment'] == 'Megahost (5+)']['review_scores_cleanliness'].dropna(),
    alternative='greater'
)
ax1.text(0.5, 4.05, f"Single vs Megahost: p < 0.0001***\n(Mann-Whitney U = {u_clean/1e6:.1f}M)", 
         ha='center', va='bottom', fontsize=9.5, bbox=dict(boxstyle='round', facecolor='#ffffcc', alpha=0.9))

sns.boxplot(
    data=data, x='host_segment', y='review_scores_communication', order=segment_order,
    hue='host_segment', legend=False,
    palette=palette_segment, ax=ax2, showfliers=False, width=0.5
)
ax2.set_title('Communication Rating by Host Segment', fontsize=12, fontweight='bold')
ax2.set_ylabel('Communication Rating', fontsize=11)
ax2.set_xlabel('')
ax2.set_ylim(4.0, 5.08)

comm_medians = data.groupby('host_segment')['review_scores_communication'].median().reindex(segment_order)
for idx, (seg, med) in enumerate(comm_medians.items()):
    ax2.text(idx, med + 0.02, f"Median: {med:.2f}", ha='center', va='bottom', fontweight='bold', color='#111111', bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

u_comm, p_comm = stats.mannwhitneyu(
    data[data['host_segment'] == 'Single-host (1)']['review_scores_communication'].dropna(),
    data[data['host_segment'] == 'Megahost (5+)']['review_scores_communication'].dropna(),
    alternative='greater'
)
ax2.text(0.5, 4.05, f"Single vs Megahost: p < 0.0001***\n(Mann-Whitney U = {u_comm/1e6:.1f}M)", 
         ha='center', va='bottom', fontsize=9.5, bbox=dict(boxstyle='round', facecolor='#ffffcc', alpha=0.9))

plt.tight_layout()
plt.savefig('../figures/study2_ratings_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Findings

### 1. Shift from sharing economy to corporate concentration
- Across the 6 analyzed cities, **megahosts (5+ listings)** control **51,60%** of the entire market. Single-listing hosts represent only **27.44%** of listings.
- Market concentration is highest in **Tokyo (73.7% megahosts)** and **Barcelona (63.1% megahosts)**.
- **Top 5% host concentration:** In Barcelona, the top 5% largest host accounts control **48.84%** of all available listings. In Madrid, the top 5% control **45.29%**.

### 2. Quality disparity: personal or anonymous host experience
- **Cleanliness rating:** Single hosts achieve a higher median cleanliness score (**4.88**) than megahosts (**4.80**). Mann-Whitney U test confirms this difference is highly statistically significant ($p < 0.0001$).
- **Communication rating:** Single hosts also outperform megahosts in communication (**4.97** vs **4.89**, $p < 0.0001$). Personal care and direct owner involvement may result in systematically higher customer satisfaction.

### 3. Key last points
- The evidence demonstrates that Airbnb in major global tourism hubs has evolved far beyond a P2P sharing economy into a professionalized real estate commercial activity.
- Anonymous corporate hosts manage larger portfolios of entire homes/apartments but suffer from lower guest satisfaction scores regarding cleanliness and personal communication compared to traditional single-property hosts.

In [ ]:
import plotly.express as px
from src.load import load_all

CITY = "Madrid"
cols = ["id", "host_id", "calculated_host_listings_count", "latitude", "longitude", "neighbourhood_cleansed"]
df = load_all(usecols=cols)
df["city_name"] = df["city"].map({k: v.name for k, v in CITIES.items()})
df = df[(df["city_name"] == CITY) & (df["calculated_host_listings_count"] >= 5)]

fig = px.density_map(
    df,
    lat="latitude",
    lon="longitude",
    radius=5,
    color_continuous_scale="YlOrRd",
    hover_data=["calculated_host_listings_count", "neighbourhood_cleansed"],
    title=f"Megahost Listing Density in {CITY} (5+ listings per host)",
    zoom=12,
    center={"lat": df["latitude"].median(), "lon": df["longitude"].median()},
    map_style="carto-positron",
)
fig.update_layout(margin=dict(l=0, r=0, t=40, b=0))
fig.show()
